# 🎓 Student Score Prediction — Feedforward Neural Network

**Dataset:** [Student Performance in Exams](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams) + [Student Performance Data Set (G1/G2 → G3)](https://www.kaggle.com/datasets/larsen0966/student-performance-data-set)

**Task:** Regression — predict a student's future (final) exam score from prior marks and demographic features.

**Architecture:** Feedforward Neural Network (Multi-layer Perceptron) built with PyTorch.

---
### Workflow
1. Install & authenticate Kaggle
2. Download datasets
3. EDA & visualisation
4. Pre-processing & feature engineering
5. Build FNN model
6. Train with early stopping
7. Evaluate (MAE, RMSE, R²)
8. Predict on new student input


## 0️⃣ Setup — Install dependencies & mount Drive

In [ ]:
# Install kaggle API
!pip install kaggle --quiet

import os, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device    : {DEVICE}')

## 1️⃣ Kaggle Authentication

> **Setup Kaggle Secrets in Colab:**
> 1. Go to https://www.kaggle.com → Account → **Create New API Token**
> 2. Open the downloaded `kaggle.json` file
> 3. In Colab: Click the 🔑 key icon (Secrets) in the left sidebar
> 4. Add two secrets:
>    - Name: `KAGGLE_USERNAME` → Value: your username from kaggle.json
>    - Name: `KAGGLE_KEY` → Value: your key from kaggle.json
> 5. Enable notebook access for both secrets

In [ ]:
from google.colab import files

print('Upload your kaggle.json ...')
uploaded = files.upload()   # <- select kaggle.json here

# Move to expected location
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json configured.')

## 2️⃣ Download Kaggle Datasets

In [ ]:
# ── Dataset 1: Students Performance in Exams (math / reading / writing scores)
!kaggle datasets download -d spscientist/students-performance-in-exams -p /data/exams --unzip -q

# ── Dataset 2: Student Performance (G1, G2 → predict G3 final grade)
!kaggle datasets download -d larsen0966/student-performance-data-set -p /data/grades --unzip -q

print('Downloads complete.')
print('\nExams dataset files:')
!ls /data/exams
print('\nGrades dataset files:')
!ls /data/grades

## 3️⃣ Load & Explore the Data

We use **both** datasets:
- **Dataset A** — `StudentsPerformance.csv`: demographic info + scores in math, reading, writing. Target = average score.
- **Dataset B** — `student-mat.csv`: period grades G1, G2 + 30 features. Target = **G3** (final grade).

In [ ]:
# ── Load Dataset A
df_a = pd.read_csv('/data/exams/StudentsPerformance.csv')
print('Dataset A shape:', df_a.shape)
df_a.head()

In [ ]:
# ── Load Dataset B
df_b = pd.read_csv('/data/grades/student-mat.csv', sep=';')
print('Dataset B shape:', df_b.shape)
df_b.head()

In [ ]:
# ── Basic stats
print('=== Dataset A — Missing values ===')
print(df_a.isnull().sum())

print('\n=== Dataset B — Missing values ===')
print(df_b.isnull().sum())

print('\n=== Dataset B — Target (G3) distribution ===')
print(df_b['G3'].describe())

### Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Dataset A — Score Distributions', fontsize=14, fontweight='bold')

for ax, col, color in zip(axes,
    ['math score', 'reading score', 'writing score'],
    ['#4C72B0', '#55A868', '#C44E52']):
    ax.hist(df_a[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df_a[col].mean(), color='black', linestyle='--', label=f'Mean={df_a[col].mean():.1f}')
    ax.set_title(col.title())
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Dataset B — G1, G2, G3 Distributions', fontsize=14, fontweight='bold')

for ax, col, color in zip(axes, ['G1', 'G2', 'G3'],
                           ['#4C72B0', '#DD8452', '#55A868']):
    ax.hist(df_b[col], bins=15, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df_b[col].mean(), color='black', linestyle='--')
    ax.set_title(f'{col} (Period {col[1]} Grade)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for Dataset B numeric features
plt.figure(figsize=(14, 10))
corr = df_b.select_dtypes(include='number').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            linewidths=0.3, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix — Dataset B', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4️⃣ Pre-processing

### 4A — Dataset A  
Target = **average of math + reading + writing scores** (future overall score)

In [ ]:
df_a = df_a.copy()

# Target: average score
df_a['avg_score'] = df_a[['math score', 'reading score', 'writing score']].mean(axis=1)

# Encode categoricals
cat_cols_a = ['gender', 'race/ethnicity', 'parental level of education',
              'lunch', 'test preparation course']
le = LabelEncoder()
for col in cat_cols_a:
    df_a[col] = le.fit_transform(df_a[col])

# Features: prior marks (math, reading) + demographics → predict writing & overall
# Input features: all except the target
FEATURES_A = ['gender', 'race/ethnicity', 'parental level of education',
               'lunch', 'test preparation course',
               'math score', 'reading score']   # prior marks as input
TARGET_A    = 'writing score'                   # predict unseen subject score

X_a = df_a[FEATURES_A].values.astype(np.float32)
y_a = df_a[TARGET_A].values.astype(np.float32)

print(f'Dataset A  —  X: {X_a.shape}, y: {y_a.shape}')
print(f'Predicting: {TARGET_A}')

### 4B — Dataset B  
Target = **G3** (final grade), inputs include G1, G2 + 30 other features

In [ ]:
df_b = df_b.copy()

# Encode binary/categorical string columns
for col in df_b.select_dtypes(include='object').columns:
    df_b[col] = LabelEncoder().fit_transform(df_b[col])

TARGET_B   = 'G3'
FEATURES_B = [c for c in df_b.columns if c != TARGET_B]

X_b = df_b[FEATURES_B].values.astype(np.float32)
y_b = df_b[TARGET_B].values.astype(np.float32)

print(f'Dataset B  —  X: {X_b.shape}, y: {y_b.shape}')
print(f'Input features ({len(FEATURES_B)}): {FEATURES_B}')

In [ ]:
# ── Choose which dataset to train on (switch here if you want Dataset A)
X_raw, y_raw, dataset_name = X_b, y_b, 'Dataset B (G1+G2 → G3)'

# Train / Validation / Test split: 70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, y_raw, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42)

# Standardise features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f'Using: {dataset_name}')
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

In [ ]:
# ── Convert to PyTorch tensors
def to_tensor(X, y):
    return (torch.tensor(X, dtype=torch.float32).to(DEVICE),
            torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(DEVICE))

X_tr, y_tr   = to_tensor(X_train, y_train)
X_vl, y_vl   = to_tensor(X_val,   y_val)
X_te, y_te   = to_tensor(X_test,  y_test)

BATCH = 32
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_vl, y_vl), batch_size=BATCH)

print(f'Batches per epoch: {len(train_loader)}')

## 5️⃣ Build the Feedforward Neural Network

In [ ]:
class StudentFNN(nn.Module):
    """
    Feedforward Neural Network for student score regression.

    Architecture:
        Input → FC(128) → BN → ReLU → Dropout(0.3)
              → FC(64)  → BN → ReLU → Dropout(0.2)
              → FC(32)  → BN → ReLU
              → FC(16)  → ReLU
              → FC(1)   (linear output — predicted score)
    """
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            # Hidden layer 1
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.30),

            # Hidden layer 2
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.20),

            # Hidden layer 3
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            # Hidden layer 4
            nn.Linear(32, 16),
            nn.ReLU(),

            # Output layer — single neuron (regression)
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)


INPUT_DIM = X_train.shape[1]
model     = StudentFNN(INPUT_DIM).to(DEVICE)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6️⃣ Train with Early Stopping

In [ ]:
criterion  = nn.MSELoss()
optimizer  = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True)

EPOCHS        = 300
PATIENCE      = 25
best_val_loss = float('inf')
patience_ctr  = 0
best_weights  = None

history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):
    # ── Training
    model.train()
    train_loss = 0.0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        pred  = model(Xb)
        loss  = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * Xb.size(0)
    train_loss /= len(train_loader.dataset)

    # ── Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for Xb, yb in val_loader:
            pred      = model(Xb)
            val_loss += criterion(pred, yb).item() * Xb.size(0)
    val_loss /= len(val_loader.dataset)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    scheduler.step(val_loss)

    if epoch % 25 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}')

    # ── Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\n⏹ Early stopping at epoch {epoch}. Best val MSE: {best_val_loss:.4f}')
            break

# Restore best weights
model.load_state_dict(best_weights)
print('\nTraining complete. Best weights restored.')

In [ ]:
# ── Loss curves
plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train MSE', color='#4C72B0')
plt.plot(history['val_loss'],   label='Val MSE',   color='#C44E52')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training & Validation Loss', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7️⃣ Evaluate on Test Set

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_te).cpu().numpy().flatten()

y_true = y_test

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)

print('=' * 40)
print('  Test Set Evaluation Results')
print('=' * 40)
print(f'  MAE  : {mae:.3f}  (avg pts off)')
print(f'  RMSE : {rmse:.3f}')
print(f'  R²   : {r2:.4f}  ({r2*100:.1f}% variance explained)')
print('=' * 40)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Actual vs Predicted
axes[0].scatter(y_true, y_pred, alpha=0.6, color='#4C72B0', edgecolors='white', s=40)
lim = [min(y_true.min(), y_pred.min()) - 1, max(y_true.max(), y_pred.max()) + 1]
axes[0].plot(lim, lim, 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Score (G3)')
axes[0].set_ylabel('Predicted Score')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Residuals
residuals = y_true - y_pred
axes[1].hist(residuals, bins=20, color='#55A868', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8️⃣ Feature Importance (Permutation-based)

In [ ]:
# Permutation importance — shuffle one feature at a time and measure RMSE increase
model.eval()
baseline_rmse = rmse
importances   = []

X_te_np = X_te.cpu().numpy()

for i in range(X_te_np.shape[1]):
    X_perm          = X_te_np.copy()
    X_perm[:, i]    = np.random.permutation(X_perm[:, i])
    X_perm_t        = torch.tensor(X_perm, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        y_perm = model(X_perm_t).cpu().numpy().flatten()
    perm_rmse = np.sqrt(mean_squared_error(y_true, y_perm))
    importances.append(perm_rmse - baseline_rmse)

feat_imp = pd.Series(importances, index=FEATURES_B).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#C44E52' if v > 0 else '#4C72B0' for v in feat_imp.values]
feat_imp.plot(kind='barh', color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('RMSE increase when feature is permuted')
plt.title('Feature Importance (Permutation)', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 10 most important features:')
print(feat_imp.head(10).to_string())

## 9️⃣ Save the Model

In [ ]:
import pickle

torch.save(model.state_dict(), 'student_fnn.pth')

# Save scaler for inference
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model saved: student_fnn.pth')
print('Scaler saved: scaler.pkl')

## 🔮 10 — Predict a New Student's Score

Fill in the values below to predict a student's **G3 (final grade)**.

> Features match the student-mat.csv columns (G1 and G2 are the most impactful — try different values!).

In [ ]:
# ─────────────────────────────────────────────────
#  ✏️  EDIT these values for a new student
# ─────────────────────────────────────────────────
new_student = {
    'school'    : 1,   # 0=GP, 1=MS
    'sex'       : 0,   # 0=F, 1=M
    'age'       : 17,
    'address'   : 1,   # 0=R(rural), 1=U(urban)
    'famsize'   : 0,   # 0=GT3 (>3), 1=LE3 (<=3)
    'Pstatus'   : 1,   # 0=A(apart), 1=T(together)
    'Medu'      : 3,   # 0-4 (mother education)
    'Fedu'      : 2,   # 0-4 (father education)
    'Mjob'      : 2,   # encoded (0-4)
    'Fjob'      : 1,
    'reason'    : 0,   # 0-3
    'guardian'  : 0,   # 0=mother,1=father,2=other
    'traveltime': 1,   # 1-4 (home→school time)
    'studytime' : 3,   # 1-4 (weekly study hours)
    'failures'  : 0,   # 0-3 (past failures)
    'schoolsup' : 0,   # 0=no, 1=yes
    'famsup'    : 1,
    'paid'      : 1,
    'activities': 0,
    'nursery'   : 1,
    'higher'    : 1,   # wants higher education
    'internet'  : 1,
    'romantic'  : 0,
    'famrel'    : 4,   # 1-5 (family relationship quality)
    'freetime'  : 3,   # 1-5
    'goout'     : 2,   # 1-5
    'Dalc'      : 1,   # 1-5 (workday alcohol)
    'Walc'      : 2,   # 1-5 (weekend alcohol)
    'health'    : 4,   # 1-5
    'absences'  : 4,   # number of absences
    'G1'        : 12,  # ← Period 1 mark
    'G2'        : 13,  # ← Period 2 mark
}
# ─────────────────────────────────────────────────

# Ensure column order matches training features
student_df  = pd.DataFrame([new_student])[FEATURES_B]
student_arr = scaler.transform(student_df.values.astype(np.float32))
student_t   = torch.tensor(student_arr, dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    predicted_score = model(student_t).item()

predicted_score = max(0, min(20, predicted_score))  # clamp to valid grade range [0,20]

print('=' * 45)
print('        🎓 Student Score Prediction')
print('=' * 45)
print(f'  Period 1 (G1) : {new_student["G1"]}')
print(f'  Period 2 (G2) : {new_student["G2"]}')
print(f'  ──────────────────────────────────')
print(f'  Predicted G3  : {predicted_score:.2f} / 20')

grade = '🟢 Pass' if predicted_score >= 10 else '🔴 At Risk'
print(f'  Status        : {grade}')
print('=' * 45)

## 🔁 Bonus — Interactive Batch Prediction

In [ ]:
# Predict for a batch of students and display results as a table
sample_df  = df_b.sample(10, random_state=7).reset_index(drop=True)
X_sample   = scaler.transform(sample_df[FEATURES_B].values.astype(np.float32))
X_sample_t = torch.tensor(X_sample, dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    preds = model(X_sample_t).cpu().numpy().flatten()

results = pd.DataFrame({
    'G1 (Period 1)' : sample_df['G1'].values,
    'G2 (Period 2)' : sample_df['G2'].values,
    'G3 Actual'     : sample_df['G3'].values,
    'G3 Predicted'  : np.clip(preds, 0, 20).round(2),
})
results['Error'] = (results['G3 Actual'] - results['G3 Predicted']).abs().round(2)
results['Status'] = results['G3 Predicted'].apply(lambda x: 'Pass' if x >= 10 else 'At Risk')

print('Batch Prediction Results')
print(results.to_string(index=False))

---
## 📝 Summary

| Component | Detail |
|---|---|
| **Dataset** | Kaggle: `larsen0966/student-performance-data-set` |
| **Task** | Regression — predict final grade G3 |
| **Architecture** | FNN: Input → 128 → 64 → 32 → 16 → 1 |
| **Regularisation** | Batch Norm + Dropout + Weight Decay |
| **Optimiser** | Adam (lr=1e-3) + ReduceLROnPlateau |
| **Early Stopping** | Patience = 25 epochs |
| **Key Inputs** | G1, G2 (prior marks) + 30 demographic features |

**To improve further:**
- Add hyperparameter tuning with Optuna
- Try Ensemble (FNN + Gradient Boosting)
- Include Dataset A for cross-subject score prediction
